In [1]:
import json
import re
from pathlib import Path
from itertools import combinations

# ====== user settings ======
CHECKPOINTS_DIR = Path("checkpoints")     # change if needed
OUT_SEQS = Path("protein_sequences.txt")
OUT_DUP_REPORT = Path("duplicates_report.txt")
FOLDER_PATTERN = re.compile(r"^(\d+)_linearfold_linearfold$")  # matches "1_linearfold_linearfold"
# ===========================

def normalize_seq(seq: str) -> str:
    # Remove whitespace/newlines just in case
    return "".join(str(seq).split())

# 1) Collect sequences in numeric index order
items = []  # list of (idx:int, folder_name:str, seq:str)
for child in CHECKPOINTS_DIR.iterdir():
    if not child.is_dir():
        continue
    m = FOLDER_PATTERN.match(child.name)
    if not m:
        continue

    idx = int(m.group(1))
    json_path = child / "training_summary.json"
    if not json_path.exists():
        # skip if missing; you can raise here if you prefer strict behavior
        continue

    try:
        data = json.loads(json_path.read_text(encoding="utf-8"))
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON: {json_path}") from e

    seq = data.get("protein_sequence")
    if not seq:
        # skip empty/missing protein_sequence
        continue

    items.append((idx, child.name, normalize_seq(seq)))

items.sort(key=lambda x: x[0])  # ensures 2 comes before 10

if not items:
    raise FileNotFoundError(
        f"No valid training_summary.json with protein_sequence found under: {CHECKPOINTS_DIR}"
    )

# 2) Write sequences to txt in the preserved order (duplicates NOT removed)
OUT_SEQS.write_text("\n".join(seq for _, _, seq in items) + "\n", encoding="utf-8")

print(f"Wrote {len(items)} sequences to {OUT_SEQS.resolve()}")
print("First few (index, folder, length):")
for idx, folder, seq in items[:5]:
    print(idx, folder, len(seq))

# 3) Detect duplicates (but do NOT modify the output order/file)
seq2idxs = {}
for idx, _, seq in items:
    seq2idxs.setdefault(seq, []).append(idx)

dups = {seq: idxs for seq, idxs in seq2idxs.items() if len(idxs) > 1}

# 4) Write duplicate report
lines = []
if not dups:
    lines.append("No duplicate protein_sequence found.")
else:
    lines.append(f"Found {len(dups)} duplicate groups.\n")

    # Group summary
    group_num = 1
    for seq, idxs in dups.items():
        lines.append(f"[Group {group_num}] indices={idxs}  length={len(seq)}")
        lines.append(seq)
        lines.append("")  # blank line
        group_num += 1

    # Pair listing
    lines.append("Duplicate index pairs (within each group):")
    for seq, idxs in dups.items():
        for a, b in combinations(idxs, 2):
            lines.append(f"{a}\t{b}")

OUT_DUP_REPORT.write_text("\n".join(lines) + "\n", encoding="utf-8")

print(f"Wrote duplicate report to {OUT_DUP_REPORT.resolve()}")
print(f"Duplicate groups: {len(dups)}")


Wrote 55 sequences to /mnt/c/CMU/carlLab/codonrl/protein_sequences.txt
First few (index, folder, length):
1 1_linearfold_linearfold 343
2 2_linearfold_linearfold 291
3 3_linearfold_linearfold 404
4 4_linearfold_linearfold 152
5 5_linearfold_linearfold 125
Wrote duplicate report to /mnt/c/CMU/carlLab/codonrl/duplicates_report.txt
Duplicate groups: 0


In [3]:
import os, sys, json, math, random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------
# USER CONFIG
# ---------------------------
# Path to your codonrl repo (so `from CodonRL_main import ...` works)
CODONRL_PATH = "."   # <-- CHANGE THIS
if CODONRL_PATH not in sys.path:
    sys.path.append(CODONRL_PATH)

# Your checkpoints folder + which trained folder to use
CHECKPOINTS_DIR = Path("checkpoints")
RUN_FOLDER = CHECKPOINTS_DIR / "21_linearfold_linearfold"   # <-- CHANGE (e.g., "12_linearfold_linearfold")

SUMMARY_JSON = RUN_FOLDER / "training_summary.json"
CKPT_PATH = RUN_FOLDER / "ckpt_best_objective.pth"

# Experiment settings
R_VALUES = [i / 10 for i in range(1, 10)]  # 0.1 ... 0.9
N_REPS_PER_R = 1                           # increase to reduce noise (e.g., 5 or 10)
RANDOM_SEED = 0

# Output
OUT_JSON = Path("divergence_experiment_results.json")

# ---------------------------
# Import the same inference utilities you already use
# (these come from your code file)  :contentReference[oaicite:2]{index=2}
# ---------------------------
# Option 1: if visualizeandbenchmark_multialpha.py is in your working directory / importable:
# import visualizeandbenchmark_multialpha as vb

# Option 2 (more robust): load it directly from the uploaded script path / your local path
SCRIPT_PATH = Path("visualizeandbenchmark_multialpha.py")  # <-- ensure this points to that file
import importlib.util
spec = importlib.util.spec_from_file_location("vb", SCRIPT_PATH)
vb = importlib.util.module_from_spec(spec)
spec.loader.exec_module(vb)

# ---------------------------
# Helpers
# ---------------------------
AA_ALPHABET = list("ACDEFGHIKLMNPQRSTVWY")

def mutate_protein(protein: str, r: float, rng: random.Random) -> str:
    """Per-position substitution with prob r. Substitution picks a random DIFFERENT AA."""
    out = []
    for aa in protein:
        if rng.random() < r:
            choices = [x for x in AA_ALPHABET if x != aa]
            out.append(rng.choice(choices))
        else:
            out.append(aa)
    return "".join(out)

def compute_performance(metrics: dict) -> float:
    """
    Default 'performance': -MFE (higher is better if MFE is less negative).
    Replace this with your true objective if you have one.
    """
    mfe = metrics.get("mfe", None)
    if mfe is None or not np.isfinite(mfe):
        return np.nan
    return -float(mfe)

def infer_once(agent, protein: str, w: dict, csc_weights: dict, mfe_calc) -> dict:
    """
    Run ONE inference (decode) and compute metrics.
    Uses your multiobjective_decode() logic.  :contentReference[oaicite:3]{index=3}
    """
    mrna = vb.multiobjective_decode(
        agent=agent,
        protein=protein,
        w=w,
        csc_weights=csc_weights,
        alpha_cai=0.5,  # <-- change if you want; or read from cfg if you store it
        alpha_csc=0.0,
        alpha_gc=0.0,
        alpha_u=0.0,
        target_gc=None,
        target_u=None,
        mfe_calc=None,
        alpha_mfe=0.0,
    )

    # Metrics (same style as your benchmark)
    cai = vb.cai_safe(mrna, w)

    vienna_mfe = None
    linearfold_mfe = None
    try:
        vienna_mfe = mfe_calc.calculate_vienna_async(mrna).result()
    except Exception:
        pass
    try:
        linearfold_mfe = mfe_calc.calculate_linearfold_async(mrna).result()
    except Exception:
        pass

    mfe = vienna_mfe if vienna_mfe is not None else linearfold_mfe

    return {
        "mrna": mrna,
        "cai": float(cai) if np.isfinite(cai) else np.nan,
        "vienna_mfe": float(vienna_mfe) if vienna_mfe is not None else None,
        "linearfold_mfe": float(linearfold_mfe) if linearfold_mfe is not None else None,
        "mfe": float(mfe) if mfe is not None else None,
        "gc": float(vb.gc_content(vb.to_dna(mrna))) if mrna else np.nan,
        "u_pct": float(vb.u_pct(vb.to_dna(mrna))) if mrna else np.nan,
    }

# ---------------------------
# Load original protein + build agent from checkpoint
# ---------------------------
assert SUMMARY_JSON.exists(), f"Missing: {SUMMARY_JSON}"
assert CKPT_PATH.exists(), f"Missing: {CKPT_PATH}"

summary = json.loads(SUMMARY_JSON.read_text(encoding="utf-8"))
original_protein = summary.get("protein_sequence", "").strip().upper()
if not original_protein:
    raise ValueError("protein_sequence not found in training_summary.json")

cfg, w = vb.load_cfg_and_w(str(SUMMARY_JSON))   # uses summ['config'] in your code :contentReference[oaicite:4]{index=4}
device = "cuda:0" if vb.torch.cuda.is_available() else "cpu"
agent = vb.build_agent(cfg, device)

sd = vb.torch.load(str(CKPT_PATH), map_location=device)
agent.policy_net.load_state_dict(sd)
agent.target_net.load_state_dict(sd)

mfe_calc = vb.get_mfe_calculator()
csc_weights = vb.load_csc_weights(None)

print("Loaded checkpoint + config.")
print("Protein length:", len(original_protein))
print("Device:", device)

# ---------------------------
# Run divergence experiment
# ---------------------------
rng_master = random.Random(RANDOM_SEED)

results = []
for r in R_VALUES:
    rep_metrics = []
    for rep in range(N_REPS_PER_R):
        # deterministically seed each replicate for reproducibility
        rng = random.Random((RANDOM_SEED + 1) * 10_000 + int(r * 1000) * 100 + rep)
        mutated = mutate_protein(original_protein, r, rng)

        metrics = infer_once(agent, mutated, w, csc_weights, mfe_calc)
        perf = compute_performance(metrics)

        rep_metrics.append({
            "r": r,
            "rep": rep,
            "mutated_protein": mutated,
            "performance": perf,
            **metrics,
        })

    # aggregate for plotting
    perfs = np.array([x["performance"] for x in rep_metrics], dtype=float)
    results.append({
        "r": r,
        "performance_mean": float(np.nanmean(perfs)),
        "performance_std": float(np.nanstd(perfs)),
        "replicates": rep_metrics,
    })

# Save raw results
OUT_JSON.write_text(json.dumps({
    "run_folder": str(RUN_FOLDER),
    "summary_json": str(SUMMARY_JSON),
    "ckpt_path": str(CKPT_PATH),
    "r_values": R_VALUES,
    "n_reps_per_r": N_REPS_PER_R,
    "random_seed": RANDOM_SEED,
    "performance_definition": "default: -MFE (Vienna if available else LinearFold)",
    "results": results,
}, indent=2), encoding="utf-8")

print(f"Saved raw results to: {OUT_JSON.resolve()}")

# ---------------------------
# Plot performance vs r
# ---------------------------
xs = [d["r"] for d in results]
ys = [d["performance_mean"] for d in results]
yerr = [d["performance_std"] for d in results]

plt.figure(figsize=(7,4))
plt.errorbar(xs, ys, yerr=yerr, marker="o", capsize=3)
plt.xlabel("Divergence rate r (per-position AA substitution prob)")
plt.ylabel("Performance (default = -MFE)")
plt.title(f"Performance vs divergence (folder: {RUN_FOLDER.name})")
plt.grid(True, alpha=0.3)
plt.savefig(f"performance_vs_divergence_{RUN_FOLDER.name}.png", dpi=300)


Initialized CachedAttentionQNetwork (Encoder-Decoder). PyTorch >= 2.0 will enable Flash Attention automatically.
Initialized CachedAttentionQNetwork (Encoder-Decoder). PyTorch >= 2.0 will enable Flash Attention automatically.
         For accurate results, provide real CSC data from mRNA stability experiments.
Loaded checkpoint + config.
Protein length: 290
Device: cpu


/tmp/ipykernel_1250/1768801564.py:130: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = vb.torch.load(str(CKPT_PATH), map_location=device)


Protein context memory has been pre-computed and cached.
Protein context memory has been pre-computed and cached.
Protein context memory has been pre-computed and cached.
Protein context memory has been pre-computed and cached.
Protein context memory has been pre-computed and cached.
Protein context memory has been pre-computed and cached.
Protein context memory has been pre-computed and cached.
Protein context memory has been pre-computed and cached.
Protein context memory has been pre-computed and cached.
Saved raw results to: /mnt/c/CMU/carlLab/codonrl/divergence_experiment_results.json
